# Wildfire dataset check: one fire, the way the model sees it

Loads a fire from the public dataset [Javyn123/wildfire-spread-2014-2025](https://huggingface.co/datasets/Javyn123/wildfire-spread-2014-2025)
and draws it with the project's own loader (`model/data.py`), so what you see is exactly what training and evaluation use.

1. **Progression**: perimeters over time, burned area, gaps between perimeters
2. **Layers**: every input layer with the final perimeter on top (misalignment or an empty layer shows here)
3. **Pairs**: every training pair (old burn, new burn, cells left out of scoring)
4. **One pair**: inputs, Rothermel spread, the physics-only forecast and a simple baseline against what happened
5. **Training crops**: random crops exactly as the data loader feeds the network
6. **A trained model's forecast** (optional, needs weights copied from ICE)

Runtime: CPU is fine. Change `FIRE` below to look at any other fire.

In [ ]:
# setup: the project code (loader, physics, plots) from GitHub
!git clone -q --depth 1 -b phase1-live-ingest https://github.com/rafiki230/WildfireRepo1.git 2>/dev/null || (cd WildfireRepo1 && git pull -q)
import sys
sys.path.insert(0, "WildfireRepo1/viz")
import fireview as V
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 90

In [ ]:
# browse the catalog: biggest perimeter series in the test years (2024-2025)
cat = V.catalog()
cat[(cat.year >= 2024) & (cat.rating == "keep")].sort_values("perimeters", ascending=False)[
    ["fire_id", "name", "year", "perimeters", "final_acres", "median_gap_h"]].head(15)

In [ ]:
FIRE = "Dragon Bravo"          # a name from the catalog (add year=2024 if the name repeats) or a fire id like "Ff1ba680f61"
d = V.fetch(FIRE)

## 1. Progression

In [ ]:
V.progression(d);

## 2. Every input layer (final perimeter on top)

In [ ]:
V.layers(d);

## 3. Every training pair
Grey = burned before, red = new burn (the target), yellow = new burn left out of scoring (far from the fire with no satellite fire nearby).

In [ ]:
V.pairs(d);

## 4. One pair exactly as training builds it
Defaults to the pair with the most growth; pass a number to pick another, e.g. `V.pair(d, 10)`. The bottom-right panels flag exactly as many cells as actually burned, so any model can be compared fairly.

In [ ]:
V.pair(d);

## 5. Training crops as the data loader feeds them

In [ ]:
V.crops(d);

## 5b. Crown fire and spotting (physics v2)
Where the canopy is likely to burn, how much that speeds up spread, fire intensity, and where embers land relative to what actually burned next.

In [ ]:
V.crownspot(d);

## 6. A trained model's forecast (optional)

On ICE, copy a model's weights and the Tessera projection somewhere you can download them, e.g.

```
cp ~/scratch/wildfire/my/runs/hybrid_v1.ckpt ~/scratch/wildfire/my/runs/tessera_pca.npz ~/
```

then download both (e.g. `scp` or the PACE OnDemand file browser) and upload them with the folder icon on the left.
`.ckpt` works while the run is still training (it uses the best weights so far). The right-hand panels show the same model after per-fire calibration on the previous pair.

In [ ]:
# V.prediction(d, 22, "hybrid_v1.ckpt", "hybrid", pca="tessera_pca.npz");
# V.prediction(d, 22, "unet_v1.pt", "unet", pca="tessera_pca.npz");